In [1]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt


In [2]:
def bs_call(S0, K, T, r, sigma):
    """欧式看涨期权的 Black-Scholes 理论价格"""
    d1 = (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S0 * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

In [3]:
def mc_call(S0, K, T, r, sigma, n_paths=100_000, seed=42):
    """用蒙特卡洛模拟给欧式看涨期权定价"""
    rng = np.random.default_rng(seed)
    Z = rng.standard_normal(n_paths)                                     # 标准正态随机数
    ST = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)  # 到期时的股价
    payoff = np.maximum(ST - K, 0)                                       # 看涨期权到期收益
    discounted = np.exp(-r * T) * payoff                                 # 折现回今天
    price = discounted.mean()
    stderr = discounted.std(ddof=1) / np.sqrt(n_paths)                   # 标准误差
    return price, stderr

In [4]:
S0, K, T, r, sigma = 100, 100, 1.0, 0.05, 0.2

bs = bs_call(S0, K, T, r, sigma)
mc, se = mc_call(S0, K, T, r, sigma)

print(f"Black-Scholes:  {bs:.4f}")
print(f"Monte Carlo:    {mc:.4f} ± {1.96*se:.4f} (95% CI)")
print(f"Difference:     {mc - bs:.4f}")

Black-Scholes:  10.4506
Monte Carlo:    10.4205 ± 0.0917 (95% CI)
Difference:     -0.0300
